In [2]:
%pip install matplotlib

  Using cached matplotlib-3.10.8-cp310-cp310-win_amd64.whl.metadata (52 kB)
  Using cached contourpy-1.3.2-cp310-cp310-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp310-cp310-win_amd64.whl.metadata (119 kB)
  Using cached kiwisolver-1.5.0-cp310-cp310-win_amd64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached matplotlib-3.10.8-cp310-cp310-win_amd64.whl (8.1 MB)
Using cached contourpy-1.3.2-cp310-cp310-win_amd64.whl (221 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.62.1-cp310-cp310-win_amd64.whl (1.6 MB)
Using cached kiwisolver-1.5.0-cp310-cp310-win_amd64.whl (73 kB)
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)

   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- --

In [1]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:

def visualize_id_truth(file_path):
    print(f"[INFO] Loading {file_path} for ID verification...")
    pcd = o3d.t.io.read_point_cloud(file_path)
    
    # 1. Get the IDs
    # Using flatten() to ensure they are 1D arrays
    c_ids = pcd.point.class_id.numpy().flatten()
    i_ids = pcd.point.instance_id.numpy().flatten()
    
    # 2. Create a "Unique Key" for every specific object
    # We combine Class and Instance into a single unique integer
    # e.g., Class 2, Instance 1 becomes 2001
    unique_keys = (c_ids * 1000) + i_ids
    
    # Get the unique objects present in the cloud
    unique_objects, indexed_ids = np.unique(unique_keys, return_inverse=True)
    num_objects = len(unique_objects)
    print(f"[INFO] Found {num_objects} unique identified objects in the cloud.")

    # 3. Generate a High-Contrast Color Palette
    # We use a colormap like 'tab20' or 'prism' which has very distinct jumping colors
    cmap = plt.get_cmap("tab20") 
    # Map each unique indexed ID to a color
    random_colors = cmap(indexed_ids % 20)[:, :3] # Cycle through 20 colors

    # 4. Apply to a legacy cloud for simple visualization
    vis_pcd = o3d.geometry.PointCloud()
    vis_pcd.points = o3d.utility.Vector3dVector(pcd.point.positions.numpy())
    vis_pcd.colors = o3d.utility.Vector3dVector(random_colors)

    print("\n--- VISUALIZATION CONTROLS ---")
    print("Each distinct color represents a unique Instance ID.")
    print("If a pier has two colors, it was split. If two piers have the same color, they were merged.")
    
    o3d.visualization.draw_geometries([vis_pcd], 
                                      window_name="ID Verification - Random Colors",
                                      width=1280, height=720)

In [3]:
def print_id_summary(pcd):
    # pcd = o3d.t.io.read_point_cloud(file_path)
    c_ids = pcd.point.class_id.numpy().flatten()
    i_ids = pcd.point.instance_id.numpy().flatten()

    unique_classes = np.unique(c_ids)
    print(f"{'Class ID':<10} | {'Instances Found':<15} | {'Points per Instance'}")
    print("-" * 50)
    for c in unique_classes:
        # Get instances belonging to this class
        instances_in_class = i_ids[c_ids == c]
        unique_inst = np.unique(instances_in_class)
        
        counts = [np.sum(instances_in_class == i) for i in unique_inst]
        print(f"{c:<10} | {len(unique_inst):<15} | {counts}")

In [9]:
pt_cld_file = r"F:\Segmentation_training\RC_bridges\training_datasets\correct_instance_labeled datasets\Culvert_Box_instance_cloud.ply"
pcd_cld = o3d.t.io.read_point_cloud(pt_cld_file)

In [8]:
o3d.visualization.draw_geometries([pcd_cld.to_legacy()], 
                                      window_name="ID Verification - Random Colors",
                                      width=1280, height=720)

In [35]:
print_id_summary(pcd_cld)

Class ID   | Instances Found | Points per Instance
--------------------------------------------------
0          | 2               | [np.int64(1303), np.int64(1314)]
1          | 1               | [np.int64(17387)]
4          | 2               | [np.int64(1225), np.int64(1179)]
5          | 4               | [np.int64(4484), np.int64(4303), np.int64(4461), np.int64(4344)]


In [10]:
visualize_id_truth(pt_cld_file)

[INFO] Loading F:\Segmentation_training\RC_bridges\training_datasets\correct_instance_labeled datasets\Culvert_Box_instance_cloud.ply for ID verification...
[INFO] Found 7 unique identified objects in the cloud.

--- VISUALIZATION CONTROLS ---
Each distinct color represents a unique Instance ID.
If a pier has two colors, it was split. If two piers have the same color, they were merged.


In [37]:
target_class_id = 4

class_ids = pcd_cld.point.class_id.cpu().numpy().flatten()
instance_ids = pcd_cld.point.instance_id.cpu().numpy().flatten()
target_mask = (class_ids == target_class_id)

target_mask_inst = instance_ids[target_mask]

unique_instances = np.unique(target_mask_inst)
unique_instances

array([0, 1], dtype=int32)

In [18]:
def mask_and_visualize(pcd, target_id):
    # 1. Ensure the class_id is a 1D mask
    # We use .as_tensor() to ensure we are working with the right object type
    class_ids = pcd.point["class_id"].reshape((-1,))
    mask = (class_ids == target_id)
    
    # 2. Create the new masked PointCloud
    masked_pcd = o3d.t.geometry.PointCloud(pcd.device)
    
    # 3. Manually list keys or use the internal dictionary to avoid 'TensorMap' error
    # Standard keys in a typical PCD are 'positions', 'colors', 'normals', 'class_id'
    available_attributes = ["positions", "colors", "instance_id", "class_id"]
    
    for key in available_attributes:
        if key in pcd.point:
            masked_pcd.point[key] = pcd.point[key][mask]
    
    # 4. Visualization logic
    num_points = masked_pcd.point.positions.shape[0]
    if num_points > 0:
        print(f"Showing Class {target_id}: {num_points} points.")
        
        # Convert to legacy for visualization
        legacy_pcd = masked_pcd.to_legacy()
        
        # Simple Visualizer
        o3d.visualization.draw_geometries(
            [legacy_pcd], 
            window_name=f"Masked Class ID: {target_id}",
            width=1280, height=720,
            left=50, top=50
        )
    else:
        print(f"Warning: No points found for Class ID {target_id}")

# --- Example Usage ---
mask_and_visualize(pcd_cld, target_id=3)

Showing Class 3: 17979 points.
